# Experimentos Grid Search — LightGBM
Exploración de 12 configuraciones sobre 6 dimensiones:
- Objetivo (tweedie, regression_l1)
- tweedie_variance_power
- is_unbalance
- Lags cortos vs largos
- max_depth
- learning_rate

## 1. Setup

In [0]:
%pip install lightgbm --quiet
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Workspace/Repos/carlos.saquel@gmail.com/santiago-weather-forecast')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.ingestion import load_from_delta_table
from src.data.preprocessing import build_features, train_test_split
from src.models.lightgbm_model import LightGBMPredictor
from src.evaluation.metrics import evaluate_model
from src.evaluation.cross_validation import TimeSeriesSplit, run_cv
from src.utils.mlflow_utils import setup_experiment, log_cv_run
from src.utils.config import *

setup_experiment()
print('✅ Setup completo')

## 2. Cargar datos

In [0]:
df_raw = load_from_delta_table('weather_raw', spark)
df = build_features(df_raw)
df_train, df_test = train_test_split(df)

print(f'\n📊 Train: {len(df_train)} días')
print(f'📊 Test:  {len(df_test)} días')

## 3. Definir grid

In [0]:
# 12 experimentos — cada uno explora una dimensión distinta
# manteniendo el resto fijo para que los resultados sean comparables

GRID = [
    # ── GRUPO A: Variaciones de objetivo ──────────────────────────
    {
        'name': 'tweedie_1.2',
        'description': 'Tweedie con variance_power bajo — más peso en ocurrencia',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.2,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },
    {
        'name': 'tweedie_1.5_baseline',
        'description': 'Tweedie baseline — referencia para comparar',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.5,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },
    {
        'name': 'tweedie_1.8',
        'description': 'Tweedie con variance_power alto — más peso en magnitud',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.8,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },
    {
        'name': 'regression_l1',
        'description': 'MAE objetivo — robusto a eventos extremos',
        'params': {
            'objective': 'regression_l1',
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },

    # ── GRUPO B: is_unbalance ──────────────────────────────────────
    {
        'name': 'tweedie_unbalance',
        'description': 'Tweedie + is_unbalance — penaliza más días lluviosos',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.5,
            'is_unbalance': True,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },
    {
        'name': 'l1_unbalance',
        'description': 'L1 + is_unbalance — combinación robustez + balance',
        'params': {
            'objective': 'regression_l1',
            'is_unbalance': True,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },

    # ── GRUPO C: Variaciones de profundidad ───────────────────────
    {
        'name': 'shallow_depth4',
        'description': 'Árbol superficial — menos sobreajuste, más generalización',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.5,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 4, 'num_leaves': 15,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },
    {
        'name': 'deep_depth10',
        'description': 'Árbol profundo — captura interacciones complejas',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.5,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 10, 'num_leaves': 63,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },

    # ── GRUPO D: Variaciones de learning rate ─────────────────────
    {
        'name': 'slow_learner',
        'description': 'Learning rate bajo + más estimadores — aprendizaje fino',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.5,
            'n_estimators': 800, 'learning_rate': 0.01,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },
    {
        'name': 'fast_learner',
        'description': 'Learning rate alto — convergencia rápida',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.5,
            'n_estimators': 200, 'learning_rate': 0.1,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },

    # ── GRUPO E: Variaciones de lags ──────────────────────────────
    # Nota: estos requieren regenerar features con distintos LAG_VARS
    # Por ahora se prueban con las features actuales (lag=1 de todo)
    # El siguiente paso será parametrizar build_features()
    {
        'name': 'tweedie_high_reg',
        'description': 'Alta regularización L1+L2 — evita sobreajuste en lags',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.5,
            'n_estimators': 300, 'learning_rate': 0.05,
            'max_depth': 6, 'num_leaves': 31,
            'min_child_samples': 30,
            'reg_alpha': 1.0, 'reg_lambda': 5.0,
            'subsample': 0.8, 'colsample_bytree': 0.8,
        }
    },
    {
        'name': 'best_combo',
        'description': 'Combinación de mejores elementos — tweedie 1.2 + unbalance + shallow',
        'params': {
            'objective': 'tweedie', 'tweedie_variance_power': 1.2,
            'is_unbalance': True,
            'n_estimators': 500, 'learning_rate': 0.05,
            'max_depth': 4, 'num_leaves': 15,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
            'reg_alpha': 0.5, 'reg_lambda': 2.0,
        }
    },
]

print(f'📋 Grid definido: {len(GRID)} experimentos')
for i, exp in enumerate(GRID):
    print(f'  {i+1:2d}. {exp["name"]:30s} — {exp["description"]}')

## 4. Ejecutar grid

In [0]:
all_results = []  # métricas promedio de cada experimento
all_cv_dfs  = []  # resultados por fold de cada experimento
run_ids     = {}  # run_id MLflow por experimento

for i, exp in enumerate(GRID):
    sep = '=' * 65
    print(f'\n{sep}')
    print(f'[{i+1}/{len(GRID)}] {exp["name"]}')
    print(f'{sep}')

    try:
        # CV
        results_cv = run_cv(
            model_class=LightGBMPredictor,
            df=df_train,
            n_splits=N_SPLITS,
            test_size=TEST_SIZE,
            min_train_size=MIN_TRAIN_SIZE,
            strategy='sliding',
            **exp['params']
        )

        # Obtener último fold de train
        cv_temp = TimeSeriesSplit(
            n_splits=N_SPLITS,
            test_size=TEST_SIZE,
            min_train_size=MIN_TRAIN_SIZE,
            strategy='sliding'
        )
        last_train, _ = cv_temp.split(df_train)[-1]

        # Reentrenar mejor modelo con último fold
        best_model = LightGBMPredictor(**best_params)
        best_model.fit(last_train)

        print(f'🏆 Evaluando en test: {best_name}')
        print(f'📅 Train: {last_train.index.min().date()} → {last_train.index.max().date()}')


        # Loggear en MLflow
        run_id = log_cv_run(
            model=model_final,
            results_df=results_cv,
            model_params=exp['params'],
            run_name=exp['name'],
            description=exp['description'],
            register_model=False,
        )
        run_ids[exp['name']] = run_id

        # Guardar métricas promedio
        avg = results_cv[results_cv['fold'] == 'promedio'].iloc[0]
        all_results.append({
            'experiment':  exp['name'],
            'objective':   exp['params'].get('objective', 'tweedie'),
            'variance_power': exp['params'].get('tweedie_variance_power', '-'),
            'is_unbalance': exp['params'].get('is_unbalance', False),
            'max_depth':   exp['params'].get('max_depth', '-'),
            'learning_rate': exp['params'].get('learning_rate', '-'),
            'mae':         float(avg.get('mae', np.nan)),
            'rmse':        float(avg.get('rmse', np.nan)),
            'r2':          float(avg.get('r2', np.nan)),
            'f1_score':    float(avg.get('f1_score', np.nan)),
            'accuracy':    float(avg.get('accuracy', np.nan)),
            'run_id':      run_id,
        })

        # Guardar resultados por fold
        results_cv['experiment'] = exp['name']
        all_cv_dfs.append(results_cv[results_cv['fold'] != 'promedio'])

        print(f"  ✅ MAE={avg.get('mae', 0):.3f}  RMSE={avg.get('rmse', 0):.3f}  F1={avg.get('f1_score', 0):.3f}")

    except Exception as e:
        print(f'  ⚠️  Error en {exp["name"]}: {str(e)}')
        continue

# Consolidar resultados
df_results  = pd.DataFrame(all_results).sort_values('rmse')
df_cv_all   = pd.concat(all_cv_dfs, ignore_index=True)

print(f'\n✅ Grid completado: {len(df_results)}/{len(GRID)} experimentos exitosos')

## 5. Comparar resultados

In [0]:
df_results

In [0]:
print('\n' + '='*65)
print('RESULTADOS GRID SEARCH — ordenados por RMSE')
print('='*65)
display(
    df_results[['experiment', 'objective', 'variance_power',
                'is_unbalance', 'max_depth', 'learning_rate',
                'mae', 'rmse', 'r2', 'f1_score']]
    .reset_index(drop=True)
    .astype(str)
)

best = df_results.iloc[0]
print(f"\n🏆 Mejor por RMSE:   {best['experiment']}  (RMSE={best['rmse']:.3f})")
best_f1 = df_results.sort_values('f1_score', ascending=False).iloc[0]
print(f"🏆 Mejor por F1:     {best_f1['experiment']}  (F1={best_f1['f1_score']:.3f})")

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Comparativa Grid Search — LightGBM', fontsize=14, fontweight='bold')

metricas = [
    ('rmse',     'RMSE (menor es mejor)',     True),
    ('mae',      'MAE (menor es mejor)',      True),
    ('f1_score', 'F1-Score (mayor es mejor)', False),
]

for ax, (metric, title, ascending) in zip(axes, metricas):
    df_plot = df_results.sort_values(metric, ascending=ascending)
    colors  = ['#e63946' if i == 0 else 'steelblue' for i in range(len(df_plot))]
    bars = ax.barh(
        df_plot['experiment'],
        df_plot[metric],
        color=colors, edgecolor='white', alpha=0.85
    )
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_title(title)
    ax.set_xlabel(metric.upper())
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [0]:
# Boxplot de RMSE por fold — muestra estabilidad de cada configuración
df_cv_all['fold'] = df_cv_all['fold'].astype(str)
experiments = df_cv_all['experiment'].unique()

fig, ax = plt.subplots(figsize=(20, 6))
data_box = [df_cv_all[df_cv_all['experiment'] == e]['rmse'].astype(float).values
            for e in experiments]

bp = ax.boxplot(
    data_box,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor='#a8dadc', color='#457b9d', linewidth=1.5),
    medianprops=dict(color='#e63946', linewidth=2),
    whiskerprops=dict(color='#457b9d', linewidth=1.5),
    capprops=dict(color='#457b9d', linewidth=1.5),
)

# Puntos individuales por fold
for i, data in enumerate(data_box):
    ax.scatter([i+1]*len(data), data, color='#e63946', alpha=0.6, s=40, zorder=3)

ax.set_xticks(range(1, len(experiments)+1))
ax.set_xticklabels(experiments, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('RMSE')
ax.set_title('Distribución de RMSE por fold — estabilidad entre configuraciones',
             fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [0]:
# Boxplot de RMSE por fold — evolución temporal de cada configuración
df_cv_plot = df_cv_all.copy()
df_cv_plot['rmse'] = df_cv_plot['rmse'].astype(float)
df_cv_plot['fold'] = df_cv_plot['fold'].astype(str)

folds = sorted(df_cv_plot['fold'].unique())
experiments = df_cv_plot['experiment'].unique()

# Paleta de colores — uno por modelo
colors = plt.cm.tab20(np.linspace(0, 1, len(experiments)))
color_map = dict(zip(experiments, colors))

fig, ax = plt.subplots(figsize=(14, 7))

data_box = [df_cv_plot[df_cv_plot['fold'] == f]['rmse'].values for f in folds]

bp = ax.boxplot(
    data_box,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor='#e0e0e0', color='#888888', linewidth=1.5),
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(color='#888888', linewidth=1.5),
    capprops=dict(color='#888888', linewidth=1.5),
)

# Un punto por modelo en cada fold
for exp in experiments:
    df_exp = df_cv_plot[df_cv_plot['experiment'] == exp].sort_values('fold')
    x_positions = [folds.index(f) + 1 for f in df_exp['fold'].values]
    ax.plot(
        x_positions,
        df_exp['rmse'].values,
        marker='o', markersize=7,
        linewidth=1.5, alpha=0.8,
        color=color_map[exp],
        label=exp
    )

ax.set_xticks(range(1, len(folds) + 1))
ax.set_xticklabels([f'Fold {f}' for f in folds], fontsize=11)
ax.set_ylabel('RMSE')
ax.set_title('Evolución de RMSE por fold — todos los modelos',
             fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Evaluar mejor modelo en test set

In [0]:
from src.utils.mlflow_utils import log_test_evaluation

# Tomar el mejor por RMSE
best_exp  = df_results.iloc[0]
best_name = best_exp['experiment']
best_params = next(e['params'] for e in GRID if e['name'] == best_name)

print(f'🏆 Evaluando en test: {best_name}')
print(f'   Parámetros: {best_params}')

# Reentrenar con train completo
best_model = LightGBMPredictor(**best_params)
best_model.fit(df_train)

# Predecir en test
preds_test = best_model.predict(df_test)
y_test     = df_test[TARGET]

# Evaluar
test_metrics = evaluate_model(y_test, preds_test, model_name=best_name)

# Loggear al run de MLflow
log_test_evaluation(
    run_id=run_ids[best_name],
    test_metrics=test_metrics,
    description='Mejor configuración del grid — evaluación en test holdout'
)

# Visualización
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle(f'{best_name} — Test Set', fontsize=14, fontweight='bold')

axes[0].plot(y_test.index, y_test.values,
             label='Real', color='steelblue', linewidth=1.5, alpha=0.8)
axes[0].plot(preds_test.index, preds_test.values,
             label='Predicción', color='#e63946', linewidth=1.5, alpha=0.8)
axes[0].set_title('Test set completo')
axes[0].set_ylabel('mm')
axes[0].legend()
axes[0].grid(alpha=0.3)

mask_invierno = (y_test.index.month >= 5) & (y_test.index.month <= 9)
axes[1].plot(y_test[mask_invierno].index, y_test[mask_invierno].values,
             label='Real', color='steelblue', linewidth=1.5, alpha=0.8)
axes[1].plot(preds_test[mask_invierno].index, preds_test[mask_invierno].values,
             label='Predicción', color='#e63946', linewidth=1.5, alpha=0.8)
axes[1].set_title('Zoom: Mayo–Septiembre (temporada de lluvia)')
axes[1].set_ylabel('mm')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
print('\n' + '='*55)
print(f'COMPARATIVA CV vs TEST — {best_name}')
print('='*55)

metricas_comparar = ['mae', 'rmse', 'r2', 'f1_score', 'accuracy']
for m in metricas_comparar:
    cv_val   = float(best_exp.get(m, np.nan))
    test_val = float(test_metrics.get(m, np.nan))
    diff     = test_val - cv_val
    simbolo  = '📈' if diff > 0 else '📉'
    print(f'  {simbolo} {m:12s}: CV={cv_val:.3f}  →  Test={test_val:.3f}  (Δ={diff:+.3f})')